# 🎮 Track 1 Starter: Building Quantum Games with Qiskit
**Qiskit Fall Fest 2026 — IIIT Kottayam**

Welcome to the Track 1 starter notebook! This notebook is designed to show you how **quantum mechanics** (superposition, entanglement, and measurement collapse) can be translated into **fun, interactive game mechanics** using Qiskit.

---

### 📌 What Makes a Game "Quantum"?
A true quantum game is **not** just a classical game with a random number generator. Instead:
1. **Superposition ($H$ gate):** Game pieces or states can be in multiple possibilities simultaneously until a player "inspects" or "attacks" them.
2. **Entanglement ($CNOT$ gate):** Actions taken on one player's piece or square instantly affect another connected piece across the board.
3. **Measurement Collapse:** Unresolved game situations collapse into definite win/loss outcomes only when a player triggers a measurement.
4. **Interference:** Players can use phase shifts or gates to cancel out bad outcomes and boost good outcomes.

## 1. Prerequisites & Setup
Make sure you have `qiskit` and `qiskit-aer` installed. Run the cell below to verify your installation.

In [ ]:
# Install necessary packages if running in a fresh environment:
# !pip install qiskit qiskit-aer matplotlib

import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler
from qiskit.visualization import plot_histogram

print("Qiskit successfully imported!")

## 2. Example 1: The Quantum Penny Flip (Meyer's Quantum Coin Game)
This classic game is from David Meyer's famous 1999 paper *"Quantum Strategies"*.

### The Rules:
- A coin starts Heads up ($|0\rangle$).
- **Turn 1 (Quantum Player):** Can apply any quantum operation (or put the coin into superposition using an $H$ gate).
- **Turn 2 (Classical Player):** Can choose to Flip the coin ($X$ gate) or Do Nothing ($I$ gate).
- **Turn 3 (Quantum Player):** Plays one final move.
- **Measurement:** The box is opened. Heads ($0$) = Quantum Player wins; Tails ($1$) = Classical Player wins.

Let's build and simulate this game to see why the Quantum player can guarantee a 100% win rate!

In [ ]:
def play_penny_flip(classical_move="FLIP"):
    """
    Simulates the Quantum Penny Flip game.
    classical_move: 'FLIP' (apply X) or 'PASS' (do nothing)
    """
    # 1 qubit (the coin), 1 classical bit (the measurement)
    qc = QuantumCircuit(1, 1)
    
    # Initial State: |0> (Heads)
    
    # --- TURN 1: Quantum Player Move ---
    # Puts coin into equal superposition (|0> + |1>) / sqrt(2)
    qc.h(0)
    qc.barrier(label="Q Move 1")
    
    # --- TURN 2: Classical Player Move ---
    if classical_move.upper() == "FLIP":
        qc.x(0)  # Classical flip
    else:
        qc.id(0) # Do nothing
    qc.barrier(label="C Move")
    
    # --- TURN 3: Quantum Player Move ---
    # Quantum player applies Hadamard again!
    # Since H * X * H = H * H = Identity, the superposition is inverted back to |0>!
    qc.h(0)
    qc.barrier(label="Q Move 2")
    
    # Final Measurement
    qc.measure(0, 0)
    
    return qc

# Test both classical choices:
qc_flip = play_penny_flip("FLIP")
print(qc_flip.draw(output="text"))

In [ ]:
# Run the simulation using StatevectorSampler
sampler = StatevectorSampler()
job = sampler.run([qc_flip], shots=1000)
result = job.result()[0]
counts = result.data.c.get_counts()

print(f"Measurement Outcomes (0 = Heads, 1 = Tails): {counts}")
if '0' in counts and counts['0'] == 1000:
    print("🎉 Quantum player won 100% of the games, regardless of what the classical player did!")

## 3. Example 2: Entangled Trapdoor / Co-op Puzzle (Entanglement Mechanic)
In this mini-game, two rooms on a spaceship are quantumly entangled.
- If Room A has an alien ($|1\rangle$), Room B is guaranteed to have an alien ($|1\rangle$).
- If Room A is safe ($|0\rangle$), Room B is safe ($|0\rangle$).
- However, neither room's state is determined until a player scouts one of the rooms!

Let's see how an interactive turn-based loop can be written:

In [ ]:
def create_entangled_board():
    qc = QuantumCircuit(2, 2)
    # Create Bell pair: (|00> + |11>) / sqrt(2)
    qc.h(0)
    qc.cx(0, 1)
    return qc

def player_inspects_room(qc, room_choice=0):
    """
    Measures the chosen room and collapses the entangled state.
    """
    qc_run = qc.copy()
    qc_run.measure(room_choice, room_choice)
    
    sampler = StatevectorSampler()
    job = sampler.run([qc_run], shots=1)
    res = job.result()[0].data.c.get_counts()
    measured_bitstring = list(res.keys())[0]
    
    # Note: Qiskit bitstring order is c1 c0 (little-endian)
    outcome = int(measured_bitstring[1 - room_choice])
    return outcome, measured_bitstring

board = create_entangled_board()
outcome, bitstring = player_inspects_room(board, room_choice=0)
state_str = "ALIEN FOUND! 👾" if outcome == 1 else "ROOM IS CLEAR! 🛡️"
print(f"Player inspected Room 0: Result = {state_str}")
print("Due to entanglement, Room 1 is now collapsed to the same fate!")

## 4. Ideas for Extending into Your Hackathon Project
Now it's your turn! You can take these basic quantum building blocks and build your own game:

1. **Quantum Battleship:**
   - Use a 4-qubit circuit (a 2x2 grid) where each qubit represents a grid location.
   - Apply $H$ or rotation gates ($RY(\theta)$) to place ships in probabilistic superposition.
   - When a player fires at a coordinate, measure that qubit! If $|1\rangle$, it's a Hit; if $|0\rangle$, it's a Miss.

2. **Quantum Tic-Tac-Toe:**
   - Players place entangled marks across two squares (e.g. $CNOT$ across two qubits).
   - When a cycle of entanglement is formed, measure the circuit to collapse the marks into classical X's and O's!

3. **Add an Interactive UI:**
   - Wrap your Python game logic with a simple CLI loop in a script or notebook.
   - Or create a simple web interface using **Streamlit** (`pip install streamlit`) or **Gradio** to make an awesome 10-minute demo video!

Happy quantum hacking!